In [13]:
import wandb
from wandb.apis.public.runs import Run
from typing import Any
import pandas as pd

In [14]:
api = wandb.Api()

In [15]:
runs = api.runs("graphcore/llama-mobile", filters={"config.name": "test-time-v2"})

In [16]:
def cleanup_run(run: Run) -> dict[str, Any]:
    d = {}
    d["torch_compile"] = True if run.config["execution"]["compile"] else False
    d["recomputation"] = run.config["execution"]["recomputation"]
    d["teacher_fsdp"] = run.config["execution"]["wrap_teacher"]
    d["trainable_centroids"] = run.config["quantisation"]["trainable_centroids"]
    d["implementation"] = run.config["quantisation"]["implementation"]
    if "step_time" in run.summary:
        d["step_time"] = run.summary["step_time"]
        d["est_t_hours"] = (100_000 / 8) * d["step_time"] / 3600
    else:
        d["step_time"] = float("inf")
        d["est_t_hours"] = float("inf")
    return d
    

In [17]:
df = pd.DataFrame([cleanup_run(run) for run in runs])
df.sort_values(by="step_time", ascending=True, inplace=True)

In [18]:
df

,torch_compile,recomputation,teacher_fsdp,trainable_centroids,implementation,step_time,est_t_hours
14,True,False,True,False,old,2.156590,7.488160
0,False,False,False,False,old,2.675572,9.290180
2,False,False,True,False,old,2.677397,9.296519
18,True,True,False,False,old,2.753211,9.559761
20,True,True,True,False,old,2.882163,10.007510
6,False,True,False,False,old,3.509391,12.185385
8,False,True,True,False,old,3.537495,12.282970
19,True,True,False,False,new,4.316782,14.988826
21,True,True,True,False,new,4.369848,15.173084
9,False,True,True,False,new,4.390585,15.245087
